# 08. 社会的ネットワーク分析 — 練習問題

**対象技術**: AI（生成AI・機械学習・自動化を含む）

社会的ネットワーク分析は、主体間の関係をネットワークとして表現し、影響力の集中や情報の伝播を分析する手法である。本ノートブックでは、AIエコシステムの主要主体のネットワークを隣接行列で定義し、固有ベクトル中心性をべき乗法で自前計算する。さらに独立カスケードモデルをモンテカルロで実装し、初期シードの置き方別に「AI安全ガイドライン採用」が広がる過程の最終採用主体数の期待値を比較する。

必要なライブラリを読み込む。

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

# --- 日本語フォント設定: 共通モジュール jp_font.py を読み込む ---
# フォント探索・登録・フォールバックの実装は repo 直下の jp_font.py に集約。
import os as _os, sys as _sys
_d = _os.path.abspath(_os.getcwd())
while not _os.path.exists(_os.path.join(_d, "jp_font.py")) and _d != _os.path.dirname(_d):
    _d = _os.path.dirname(_d)
_sys.path.insert(0, _d)
from jp_font import setup_japanese_font
setup_japanese_font()


AIエコシステムの主体（ノード）を定義する。

In [ ]:
# AIエコシステムの主体（ノード）
NODES = [
    "巨大IT企業", "専業AIスタートアップ", "OSSコミュニティ",
    "クラウド事業者", "大学研究室", "規制当局", "導入企業",
]

# 図のラベル用に英数字IDを用意（日本語の文字化け回避）
NODE_IDS = ["BigTech", "Startup", "OSS", "Cloud", "Univ", "Regulator", "Adopter"]

主体間関係（モデル提供/人材流動/共同研究/依存関係）を表す無向隣接行列を構築する。

In [ ]:
def build_adjacency():
    """主体間関係（モデル提供/人材流動/共同研究/依存）の無向隣接行列を返す。"""
    n = len(NODES)
    A = np.zeros((n, n))
    # (i, j) のペアで関係を定義（無向なので対称化）
    edges = [
        (0, 1), (0, 2), (0, 3), (0, 4),   # 巨大ITは広くつながる
        (1, 2), (1, 4), (1, 3),           # スタートアップ-OSS/大学/クラウド
        (2, 4), (2, 6),                   # OSS-大学/導入企業
        (3, 6),                           # クラウド-導入企業
        (5, 0), (5, 6),                   # 規制当局-巨大IT/導入企業
        (4, 5),                           # 大学-規制当局
    ]
    for i, j in edges:
        A[i, j] = A[j, i] = 1.0
    return A

固有ベクトル中心性をべき乗法で自前計算する関数を定義する。隣接行列を繰り返し作用させ、最大固有値方向のベクトルへ収束させる。

In [ ]:
def eigenvector_centrality(A, n_iter=200, tol=1e-9):
    """べき乗法で固有ベクトル中心性を自前計算する。"""
    n = A.shape[0]
    x = np.ones(n) / n              # 初期ベクトル
    for _ in range(n_iter):
        x_new = A @ x               # 隣接行列を作用させる
        norm = np.linalg.norm(x_new)
        if norm == 0:
            break
        x_new = x_new / norm        # 正規化（最大固有値方向へ収束）
        if np.linalg.norm(x_new - x) < tol:
            x = x_new
            break
        x = x_new
    return x

独立カスケードモデルを定義する。各採用済み主体は隣接する未採用主体を確率 prob で「1回だけ」採用させようと試みる。ここでの「採用」は AI安全ガイドラインの採択を表す。

In [ ]:
def independent_cascade(A, seeds, prob, rng):
    """独立カスケードモデルを1回実行し、最終採用主体の集合を返す。"""
    n = A.shape[0]
    active = set(seeds)             # 採用済み主体
    frontier = list(seeds)          # 今期に新規採用した主体
    while frontier:
        new_frontier = []
        for u in frontier:
            # u は隣接する未採用主体を確率 prob で採用させようと「1回だけ」試みる
            for v in range(n):
                if A[u, v] > 0 and v not in active:
                    if rng.random() < prob:
                        active.add(v)
                        new_frontier.append(v)
        frontier = new_frontier
    return active


def expected_spread(A, seeds, prob, n_trials, rng):
    """モンテカルロで独立カスケードの最終採用主体数の期待値を求める。"""
    total = 0
    for _ in range(n_trials):
        total += len(independent_cascade(A, seeds, prob, rng))
    return total / n_trials

ネットワークを構築し、固有ベクトル中心性と次数を計算して中心性の高い順に表示する。

In [ ]:
rng = np.random.default_rng(7)
A = build_adjacency()

# --- 固有ベクトル中心性 ---
ec = eigenvector_centrality(A)
degree = A.sum(axis=1)
print("=" * 60)
print("AIエコシステムネットワーク : 中心性分析")
print("=" * 60)
print(f"{'主体':<22}{'次数':>8}{'固有ベクトル中心性':>20}")
print("-" * 60)
for i in np.argsort(-ec):
    print(f"{NODES[i]:<22}{int(degree[i]):>8}{ec[i]:>20.4f}")
print()
print(f"[最高中心性] {NODES[int(np.argmax(ec))]} "
      "— 多数の主体と結ばれ影響力が集中")

独立カスケードモデルで「AI安全ガイドライン採用」の普及をシミュレーションし、初期シードの置き方を 3 通り比較する。

In [ ]:
prob, n_trials = 0.4, 3000
print(f"{'='*60}")
print(f"独立カスケードによるAI安全ガイドライン普及シミュレーション "
      f"(伝播確率={prob}, {n_trials}試行)")
print("=" * 60)
scenarios = {
    "規制当局のみ": [5],
    "巨大IT企業のみ": [0],
    "規制当局 + 巨大IT企業": [5, 0],
}
for name, seeds in scenarios.items():
    spread = expected_spread(A, seeds, prob, n_trials, rng)
    print(f"  シード={name:<22} → 最終採用主体数の期待値 "
          f"{spread:.2f} / {len(NODES)}")

print()
print("[解釈] 規制当局と巨大IT企業を同時にシードに置くと最終")
print("       採用数が最大化される。AI安全ガイドラインの普及は")
print("       制度主導と市場主導の二正面で進めるべき、という")
print("       政策的含意が得られる。")

## 可視化: 主体ネットワーク図と採用拡大カーブ

左図は主体ネットワーク。ノードを円周上に配置し、エッジを線で結ぶ。ノードサイズは固有ベクトル中心性に対応させる。右図は「規制当局 + 巨大IT企業」をシードとした独立カスケードを 1 回実行したときの採用主体数の時間発展（採用拡大カーブ）を、複数試行ぶん重ねて描く。

In [ ]:
import math

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

# --- 左: ネットワーク図 ---
ax = axes[0]
n = len(NODES)
pos = {i: (math.cos(2 * math.pi * i / n), math.sin(2 * math.pi * i / n))
       for i in range(n)}
# エッジ
for i in range(n):
    for j in range(i + 1, n):
        if A[i, j] > 0:
            ax.plot([pos[i][0], pos[j][0]], [pos[i][1], pos[j][1]],
                    color="gray", lw=1.0, zorder=1)
# ノード（サイズ = 固有ベクトル中心性）
sizes = 300 + ec / ec.max() * 2500
sc = ax.scatter([pos[i][0] for i in range(n)],
                [pos[i][1] for i in range(n)],
                s=sizes, c=ec, cmap="viridis", zorder=2,
                edgecolors="black")
for i in range(n):
    ax.annotate(NODE_IDS[i], pos[i], ha="center", va="center",
                fontsize=8, zorder=3,
                xytext=(0, 0), textcoords="offset points")
ax.set_title("AI ecosystem network (node size/color = eigenvector centrality)")
ax.set_aspect("equal")
ax.axis("off")
fig.colorbar(sc, ax=ax, fraction=0.046, label="eigenvector centrality")


# --- 右: 独立カスケードの採用拡大カーブ ---
def cascade_curve(A, seeds, prob, rng):
    """1回のカスケードで、各ステップの累積採用主体数の系列を返す。"""
    n = A.shape[0]
    active = set(seeds)
    frontier = list(seeds)
    curve = [len(active)]
    while frontier:
        new_frontier = []
        for u in frontier:
            for v in range(n):
                if A[u, v] > 0 and v not in active:
                    if rng.random() < prob:
                        active.add(v)
                        new_frontier.append(v)
        frontier = new_frontier
        curve.append(len(active))
    return curve

ax = axes[1]
rng2 = np.random.default_rng(20)
max_len = 0
curves = []
for _ in range(40):
    c = cascade_curve(A, [5, 0], prob, rng2)
    curves.append(c)
    max_len = max(max_len, len(c))
# 各曲線を最終値で右側に伸ばして揃える
padded = []
for c in curves:
    cc = c + [c[-1]] * (max_len - len(c))
    padded.append(cc)
    ax.plot(range(max_len), cc, color="steelblue", alpha=0.25, lw=1)
padded = np.array(padded)
ax.plot(range(max_len), padded.mean(axis=0), color="navy", lw=2.5,
        label="mean over 40 runs")
ax.set_xlabel("Cascade step (time)")
ax.set_ylabel("Number of adopting agents")
ax.set_title("Independent cascade adoption growth (seed: Regulator + BigTech)")
ax.set_xticks(range(max_len))
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 未来デザイン論文での使われ方と結論への影響

未来デザインの論文において社会的ネットワーク分析は、技術や政策をめぐる主体——企業、研究機関、規制当局、ユーザー集団——を結節点とし、その関係構造の上で中心性や情報・採用の伝播を分析するために用いられる。論文の典型的な使われ方は、誰が変化の鍵を握るキーストーン主体か、どこが普及を律速する伝播ボトルネックかを特定し、「働きかけるべき相手はここだ」と論じることである。したがってこの手法が生み出す結論は、「何が起きるか」よりも「誰が・どこが要か」に答える型をとる。

結論の型は特定の主体や経路の名指しに収束しやすい。境界設定の面では、ネットワークに含めた主体と関係しか分析対象にならず、図に描かれなかったアクターは構造的に不可視となる。時間観の面では、未来は与えられた構造の上を情報や採用が伝わっていく過程として描かれ、未来の形は現在の関係配置によって大きく規定されるとみなされる。価値の所在は、どの中心性指標を「重要さ」の代理とするか、どの関係を辺として描くかという選択に埋め込まれ、指標を変えれば要となる主体も入れ替わる。

最大の限界は、多くの分析が静的なネットワークを前提とすることである。現実には、技術の進展そのものが主体間の関係を組み替え、新たな結節点を生み、古い結びつきを断つ。関係構造を固定したまま伝播を論じると、この関係自体の変化が結論から抜け落ちる。結果として、この手法に依拠した論文の結論は、現在の構造のもとで誰が要かを鮮明に示せる一方、その構造が将来も続くという暗黙の仮定の上に立っている。要点の名指しは、構造の固定という条件つきで読む必要がある。

## 発展課題

**課題A**: 「最終採用数を最大にするシード集合」を貪欲法で選べ。各ステップで『追加したとき最終採用数の増分が最大の主体』を1つずつシードに加える。中心性上位を選ぶ素朴な戦略と比較する。

**課題B**: 媒介中心性（betweenness）を自前実装せよ。全ノード対の最短経路（幅優先探索）を求め、各ノードが何本の最短経路上に乗るかを数える。固有ベクトル中心性との順位の違いを観察する。